In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# Configuración base
# ============================================================

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

FILES = {
    "01_candidatos_normalizados": ROOT / "data/processed/met_candidatos_normalizados.csv",
    "02_aceptados_filtro_textil": ROOT / "data/interim/met_curacion_previa/met_textiles_aceptados_preliminar.csv",
    "03_rechazados_filtro_textil": ROOT / "data/processed/met_textiles_rejected.csv",
    "04_corpus_preliminar": ROOT / "data/interim/met_curacion_previa/corpus_preliminar.csv",
    "05_corpus_principal_previo": ROOT / "data/interim/met_curacion_previa/corpus_principal_previo.csv",
    "06_corpus_complementario_previo": ROOT / "data/interim/met_curacion_previa/corpus_complementario_previo.csv",
    "07_corpus_duplicados_previo": ROOT / "data/processed/corpus_met_textiles_andinos_v1_duplicados.csv",
}

def read_csv(path):
    if not path.exists():
        print(f"No existe: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

dfs = {name: read_csv(path) for name, path in FILES.items()}

resumen = []

for name, df in dfs.items():
    resumen.append({
        "archivo": name,
        "filas": len(df),
        "columnas": len(df.columns),
        "tiene_id_objeto": "id_objeto" in df.columns,
        "tiene_url_imagen": any(c in df.columns for c in ["url_imagen", "primaryImage", "primaryImageSmall"]),
        "columnas_motivo": [c for c in df.columns if "motivo" in c.lower()],
    })

pd.DataFrame(resumen)

In [ ]:
# ============================================================
# Auditoría de pasos del pipeline
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

def get_ids(df):
    if df.empty or "id_objeto" not in df.columns:
        return set()
    return set(df["id_objeto"].apply(norm_id))

ids = {name: get_ids(df) for name, df in dfs.items()}

transiciones = [
    ("01_candidatos_normalizados", "02_aceptados_filtro_textil"),
    ("01_candidatos_normalizados", "03_rechazados_filtro_textil"),
    ("02_aceptados_filtro_textil", "04_corpus_preliminar"),
    ("04_corpus_preliminar", "05_corpus_principal_previo"),
    ("04_corpus_preliminar", "06_corpus_complementario_previo"),
    ("04_corpus_preliminar", "07_corpus_duplicados_previo"),
]

rows = []

for origen, destino in transiciones:
    origen_ids = ids[origen]
    destino_ids = ids[destino]
    
    rows.append({
        "origen": origen,
        "destino": destino,
        "ids_origen": len(origen_ids),
        "ids_destino": len(destino_ids),
        "ids_en_comun": len(origen_ids & destino_ids),
        "ids_que_no_pasaron": len(origen_ids - destino_ids),
    })

pd.DataFrame(rows)

In [ ]:
# ============================================================
# Distribución de motivos por archivo
# ============================================================

for name, df in dfs.items():
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    motivo_cols = [c for c in df.columns if "motivo" in c.lower()]
    
    if not motivo_cols:
        print("No tiene columnas de motivo.")
        continue
    
    for col in motivo_cols:
        print(f"\nColumna: {col}")
        display(df[col].fillna("(vacío)").value_counts().reset_index().rename(
            columns={"index": col, col: "conteo"}
        ))

In [ ]:
# ============================================================
# Identificar registros aceptados que NO pasaron al corpus preliminar
# ============================================================

aceptados = dfs["02_aceptados_filtro_textil"].copy()
corpus_preliminar = dfs["04_corpus_preliminar"].copy()

ids_preliminar = get_ids(corpus_preliminar)

aceptados_no_preliminar = aceptados[
    ~aceptados["id_objeto"].apply(norm_id).isin(ids_preliminar)
].copy()

print("Aceptados por filtro textil:", len(aceptados))
print("Pasaron a corpus preliminar:", len(corpus_preliminar))
print("Aceptados que NO pasaron a corpus preliminar:", len(aceptados_no_preliminar))

cols_revision = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "motivo_filtrado",
    "url_objeto",
    "url_imagen",
    "primaryImage",
    "primaryImageSmall",
]

cols_revision = [c for c in cols_revision if c in aceptados_no_preliminar.columns]

aceptados_no_preliminar[cols_revision].head(20)

In [ ]:
# ============================================================
# Resumen de los aceptados que no pasaron al corpus preliminar
# ============================================================

for col in [
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "cultura",
]:
    if col in aceptados_no_preliminar.columns:
        print("\n" + "=" * 80)
        print(col)
        print("=" * 80)
        display(
            aceptados_no_preliminar[col]
            .fillna("(vacío)")
            .value_counts()
            .head(30)
            .reset_index()
            .rename(columns={"index": col, col: "conteo"})
        )

In [ ]:
# ============================================================
# Crear galería visual de aceptados que NO pasaron a corpus preliminar
# ============================================================

from pathlib import Path
import html

OUTPUT_DIR = ROOT / "outputs/archive/review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_image_col(df):
    candidates = [
        "url_imagen",
        "image_url",
        "imagen_url",
        "primaryImageSmall",
        "primaryImage",
        "url_imagen_principal",
    ]
    for col in candidates:
        if col in df.columns:
            return col
    return None

def find_object_url_col(df):
    candidates = ["url_objeto", "objectURL", "url", "link"]
    for col in candidates:
        if col in df.columns:
            return col
    return None

image_col = find_image_col(aceptados_no_preliminar)
object_url_col = find_object_url_col(aceptados_no_preliminar)

print("Columna de imagen detectada:", image_col)
print("Columna de URL objeto detectada:", object_url_col)

def safe_value(row, col):
    if col is None or col not in row.index:
        return ""
    value = row[col]
    if pd.isna(value):
        return ""
    return str(value)

def build_gallery(df, output_path, title):
    cards = []

    for _, row in df.iterrows():
        img = safe_value(row, image_col)
        obj_url = safe_value(row, object_url_col)

        object_id = safe_value(row, "id_objeto")
        titulo = safe_value(row, "titulo_original")
        cultura = safe_value(row, "cultura")
        periodo = safe_value(row, "periodo")
        clasificacion = safe_value(row, "clasificacion_original")
        nombre_objeto = safe_value(row, "nombre_objeto_original")
        material = safe_value(row, "material_original")
        motivo = safe_value(row, "motivo_filtrado")

        link_html = ""
        if obj_url:
            link_html = f'<p><a href="{html.escape(obj_url)}" target="_blank">Ver objeto en museo</a></p>'

        img_html = ""
        if img:
            img_html = f'<img src="{html.escape(img)}" loading="lazy">'
        else:
            img_html = '<div class="no-image">Sin imagen detectada</div>'

        card = f"""
        <div class="card">
            {img_html}
            <div class="info">
                <h3>{html.escape(object_id)} - {html.escape(titulo)}</h3>
                <p><b>Cultura:</b> {html.escape(cultura)}</p>
                <p><b>Periodo:</b> {html.escape(periodo)}</p>
                <p><b>Clasificación:</b> {html.escape(clasificacion)}</p>
                <p><b>Objeto:</b> {html.escape(nombre_objeto)}</p>
                <p><b>Material:</b> {html.escape(material)}</p>
                <p><b>Motivo actual:</b> {html.escape(motivo)}</p>
                {link_html}
                <p><b>Revisión manual:</b> ______________________________</p>
                <p><b>Decisión sugerida:</b> [ ] pasa a corpus principal | [ ] secundario | [ ] descartar</p>
            </div>
        </div>
        """
        cards.append(card)

    html_text = f"""
    <!doctype html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>{html.escape(title)}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 24px;
                background: #f6f6f6;
            }}
            h1 {{
                margin-bottom: 8px;
            }}
            .summary {{
                margin-bottom: 24px;
                color: #333;
            }}
            .grid {{
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));
                gap: 18px;
            }}
            .card {{
                background: white;
                border: 1px solid #ddd;
                border-radius: 8px;
                overflow: hidden;
            }}
            img {{
                width: 100%;
                height: 280px;
                object-fit: contain;
                background: #eee;
                border-bottom: 1px solid #ddd;
            }}
            .no-image {{
                height: 280px;
                display: flex;
                align-items: center;
                justify-content: center;
                background: #ddd;
                color: #555;
            }}
            .info {{
                padding: 12px;
                font-size: 14px;
            }}
            h3 {{
                font-size: 15px;
                margin-top: 0;
            }}
            p {{
                margin: 6px 0;
            }}
        </style>
    </head>
    <body>
        <h1>{html.escape(title)}</h1>
        <p class="summary">Total registros: {len(df)}</p>
        <div class="grid">
            {''.join(cards)}
        </div>
    </body>
    </html>
    """

    output_path.write_text(html_text, encoding="utf-8")
    return output_path

gallery_path = build_gallery(
    aceptados_no_preliminar,
    OUTPUT_DIR / "auditoria_aceptados_no_preliminar.html",
    "Auditoría visual - Aceptados que no pasaron a corpus preliminar"
)

gallery_path

In [ ]:
gallery_path

In [ ]:
# ============================================================
# Clasificación preliminar para auditar los 161 aceptados no preliminar
# ============================================================

def text_norm(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

def clasificar_revision(row):
    nombre = text_norm(row.get("nombre_objeto_original", ""))
    titulo = text_norm(row.get("titulo_original", ""))
    clasif = text_norm(row.get("clasificacion_original", ""))
    material = text_norm(row.get("material_original", ""))

    texto = " ".join([nombre, titulo, clasif, material])

    # Materiales o implementos textiles, pero no superficie visual
    if "skein" in texto:
        return "descartar_material_sin_superficie", "madeja/hilo sin superficie visual útil"

    if "basket" in texto or "implement" in texto:
        return "descartar_implemento_textil", "implemento asociado al tejido, no textil analizable"

    # Featherwork: importante, pero conviene separarlo del corpus principal
    if "feather" in texto:
        return "secundario_featherwork", "textil con plumas / featherwork, revisar como corpus secundario"

    # Objetos estrechos o longitudinales
    if any(k in texto for k in ["sash", "headband", "band", "belt"]):
        return "revision_formato_longitudinal", "formato longitudinal; revisar si contiene motivos útiles"

    # Prendas pequeñas o tridimensionales
    if any(k in texto for k in ["hat", "cap", "mask", "headdress"]):
        return "revision_prenda_accesorio", "prenda/accesorio; revisar si aporta iconografía o patrón"

    # Posibles piezas principales para el corpus
    if any(k in texto for k in [
        "textile fragment",
        "mantle fragment",
        "mantle",
        "bag",
        "panel",
        "tunic",
        "tabard",
        "hanging",
        "fragment",
        "border"
    ]):
        return "candidato_corpus_principal", "posible superficie textil útil; requiere revisión visual"

    return "revision_manual", "caso no cubierto por reglas preliminares"

aceptados_no_preliminar[["categoria_revision", "motivo_revision_preliminar"]] = aceptados_no_preliminar.apply(
    lambda row: pd.Series(clasificar_revision(row)),
    axis=1
)

display(
    aceptados_no_preliminar["categoria_revision"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "categoria_revision", "categoria_revision": "conteo"})
)

In [ ]:
# ============================================================
# Ver ejemplos por categoría preliminar
# ============================================================

cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "categoria_revision",
    "motivo_revision_preliminar",
    "url_objeto",
    "url_imagen",
]

cols = [c for c in cols if c in aceptados_no_preliminar.columns]

for categoria in aceptados_no_preliminar["categoria_revision"].unique():
    print("\n" + "=" * 100)
    print(categoria)
    print("=" * 100)
    display(
        aceptados_no_preliminar.loc[
            aceptados_no_preliminar["categoria_revision"] == categoria,
            cols
        ].head(15)
    )

In [ ]:
# ============================================================
# Lote 1: posibles recuperables para corpus principal
# ============================================================

lote_01 = aceptados_no_preliminar[
    aceptados_no_preliminar["categoria_revision"] == "candidato_corpus_principal"
].copy()

print("Registros en lote 01:", len(lote_01))

gallery_path = build_gallery(
    lote_01,
    OUTPUT_DIR / "auditoria_lote_01_candidatos_corpus_principal.html",
    "Lote 01 - Posibles recuperables para corpus principal"
)

gallery_path

In [ ]:
# ============================================================
# Clasificación preliminar mejorada para auditoría
# ============================================================

def text_norm(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

def contiene(texto, palabras):
    return any(p in texto for p in palabras)

def clasificar_revision_v2(row):
    nombre = text_norm(row.get("nombre_objeto_original", ""))
    titulo = text_norm(row.get("titulo_original", ""))
    clasif = text_norm(row.get("clasificacion_original", ""))
    material = text_norm(row.get("material_original", ""))
    cultura = text_norm(row.get("cultura", ""))

    texto = " ".join([nombre, titulo, clasif, material, cultura])

    # 1. Casos sin superficie textil analizable
    if "skein" in nombre or "skein" in titulo:
        return "descartar_material_sin_superficie", "madeja/hilo: material textil, pero sin superficie visual para análisis"

    if "basket" in nombre or "implement" in clasif or "implement" in nombre:
        return "descartar_implemento_textil", "implemento asociado al tejido, no pieza textil analizable"

    # 2. Featherwork: textil válido, pero corpus secundario por materialidad especial
    if "feather" in texto:
        return "secundario_featherwork", "textil con plumas; conservar en corpus secundario"

    # 3. Piezas principales: prioridad alta
    # Esto corrige casos como 'Tunic with Diamond Band'
    if contiene(nombre, ["tunic", "mantle", "panel", "bag", "tabard", "hanging"]):
        return "candidato_corpus_principal", "pieza textil con superficie o composición potencialmente útil"

    if contiene(titulo, ["tunic", "mantle", "panel", "bag", "tabard", "hanging"]):
        return "candidato_corpus_principal", "título indica pieza textil principal con superficie útil"

    # 4. Fragmentos textiles: revisar visualmente, muchos pueden recuperarse
    if contiene(nombre, ["textile fragment", "mantle fragment", "tunic fragment", "bag fragment", "fragment"]):
        return "candidato_corpus_principal", "fragmento textil potencialmente útil; requiere revisión visual"

    # 5. Bordes y bandas: no descartarlos automáticamente
    if contiene(nombre, ["border fragment", "border fragments"]) or contiene(titulo, ["border fragment", "border fragments"]):
        return "revision_borde_iconografico", "borde/fragmento con posible iconografía; revisar legibilidad visual"

    # 6. Formatos longitudinales
    if contiene(nombre, ["sash", "headband", "band", "belt"]) or contiene(titulo, ["sash", "headband", "band fragment", "belt"]):
        return "revision_formato_longitudinal", "formato longitudinal; revisar si aporta motivos útiles"

    # 7. Prendas/accesorios con posible valor iconográfico
    if contiene(nombre, ["hat", "cap", "mask", "headdress"]) or contiene(titulo, ["hat", "cap", "mask", "headdress"]):
        if contiene(cultura, ["wari", "tiwanaku"]):
            return "revision_prenda_iconografica_prioritaria", "prenda Wari/Tiwanaku con posible valor iconográfico"
        return "revision_prenda_accesorio", "prenda/accesorio textil; revisar si aporta patrón o iconografía"

    # 8. Hondas y artefactos textiles
    if contiene(nombre, ["sling", "sling shot"]) or contiene(titulo, ["sling", "sling shot"]):
        return "secundario_artefacto_textil", "artefacto textil funcional; conservar como secundario o revisar aparte"

    return "revision_manual", "caso no cubierto por reglas preliminares"

aceptados_no_preliminar[["categoria_revision_v2", "motivo_revision_v2"]] = aceptados_no_preliminar.apply(
    lambda row: pd.Series(clasificar_revision_v2(row)),
    axis=1
)

display(
    aceptados_no_preliminar["categoria_revision_v2"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "categoria_revision_v2", "categoria_revision_v2": "conteo"})
)

In [ ]:
# ============================================================
# Ejemplos por categoría mejorada
# ============================================================

cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "url_objeto",
    "url_imagen",
]

cols = [c for c in cols if c in aceptados_no_preliminar.columns]

for categoria in aceptados_no_preliminar["categoria_revision_v2"].unique():
    print("\n" + "=" * 100)
    print(categoria)
    print("=" * 100)
    display(
        aceptados_no_preliminar.loc[
            aceptados_no_preliminar["categoria_revision_v2"] == categoria,
            cols
        ].head(20)
    )

In [ ]:
# ============================================================
# Generar galerías por categoría mejorada
# ============================================================

categorias_a_revisar = [
    "candidato_corpus_principal",
    "revision_borde_iconografico",
    "revision_prenda_iconografica_prioritaria",
    "revision_formato_longitudinal",
    "revision_prenda_accesorio",
    "secundario_featherwork",
    "secundario_artefacto_textil",
]

for categoria in categorias_a_revisar:
    lote = aceptados_no_preliminar[
        aceptados_no_preliminar["categoria_revision_v2"] == categoria
    ].copy()

    if len(lote) == 0:
        continue

    path = build_gallery(
        lote,
        OUTPUT_DIR / f"auditoria_{categoria}.html",
        f"Auditoría visual - {categoria}"
    )

    print(categoria, len(lote), path)

In [ ]:
# ============================================================
# Crear tabla de auditoría manual para los 161 aceptados no preliminar
# ============================================================

audit_cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "motivo_filtrado",
    "categoria_revision_v2",
    "motivo_revision_v2",
]

audit_cols = [c for c in audit_cols if c in aceptados_no_preliminar.columns]

auditoria_aceptados_no_preliminar = aceptados_no_preliminar[audit_cols].copy()

# Columnas nuevas para completar después de la revisión visual
auditoria_aceptados_no_preliminar["decision_auditoria"] = ""
auditoria_aceptados_no_preliminar["motivo_auditoria"] = ""
auditoria_aceptados_no_preliminar["observacion_auditoria"] = ""

# Sugerencia automática inicial, no definitiva
def sugerir_decision(row):
    cat = row["categoria_revision_v2"]

    if cat == "candidato_corpus_principal":
        return "revisar_para_corpus_principal"

    if cat == "revision_prenda_iconografica_prioritaria":
        return "revisar_para_corpus_principal_o_secundario"

    if cat == "revision_formato_longitudinal":
        return "revisar_para_secundario_o_principal"

    if cat == "revision_prenda_accesorio":
        return "revisar_para_secundario"

    if cat == "secundario_featherwork":
        return "pasar_a_corpus_secundario"

    if cat == "secundario_artefacto_textil":
        return "pasar_a_corpus_secundario"

    if cat == "descartar_material_sin_superficie":
        return "descartar"

    if cat == "descartar_implemento_textil":
        return "descartar"

    return "revisar"

auditoria_aceptados_no_preliminar["decision_sugerida"] = auditoria_aceptados_no_preliminar.apply(
    sugerir_decision,
    axis=1
)

output_csv = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)

auditoria_aceptados_no_preliminar.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("Archivo generado:")
print(output_csv)

display(
    auditoria_aceptados_no_preliminar[
        [
            "id_objeto",
            "titulo_original",
            "cultura",
            "nombre_objeto_original",
            "categoria_revision_v2",
            "decision_sugerida",
            "decision_auditoria",
            "motivo_auditoria",
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# Resumen de decisiones sugeridas
# ============================================================

display(
    auditoria_aceptados_no_preliminar["decision_sugerida"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "decision_sugerida", "decision_sugerida": "conteo"})
)

In [ ]:
# ============================================================
# Exportar auditoría editable
# ============================================================

audit_path = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar2.xlsx"

cols_export = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "decision_sugerida",
    "decision_auditoria",
    "motivo_auditoria",
    "observacion_auditoria",
]

cols_export = [c for c in cols_export if c in auditoria_aceptados_no_preliminar.columns]

auditoria_aceptados_no_preliminar[cols_export].to_excel(audit_path, index=False)

print(audit_path)

In [ ]:
# ============================================================
# Exportar auditoría editable con links activos
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font

audit_path = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar.xlsx"

cols_export = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "decision_sugerida",
    "decision_auditoria",
    "motivo_auditoria",
    "observacion_auditoria",
]

cols_export = [c for c in cols_export if c in auditoria_aceptados_no_preliminar.columns]

auditoria_aceptados_no_preliminar[cols_export].to_excel(audit_path, index=False)

# Reabrir el Excel para convertir columnas I y J en hipervínculos
wb = load_workbook(audit_path)
ws = wb.active

for col in ["H", "I"]:
    for row in range(2, ws.max_row + 1):  # desde fila 2 para no tocar encabezados
        cell = ws[f"{col}{row}"]

        if cell.value and str(cell.value).startswith("http"):
            cell.hyperlink = cell.value
            cell.font = Font(color="0000FF", underline="single")

wb.save(audit_path)

print(audit_path)

In [ ]:
# ============================================================
# Validar decisiones y motivos insertados
# ============================================================

audit_path = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar.xlsx"
audit = pd.read_excel(audit_path)

DECISIONES_VALIDAS = {
    "pasar_a_corpus_principal",
    "pasar_a_corpus_secundario",
    "descartar",
    "requiere_revision",
}

MOTIVOS_VALIDOS = {
    "superficie_textil_util",
    "fragmento_textil_util",
    "iconografia_legible",
    "composicion_visual_util",
    "iconografia_wari_tiwanaku_relevante",
    "patron_geometrico_legible",
    "prenda_tridimensional_util",
    "prenda_tridimensional_no_prioritaria",
    "formato_longitudinal_con_motivos",
    "formato_longitudinal_sin_motivos",
    "faja_o_banda_util_secundario",
    "featherwork_secundario",
    "artefacto_textil_secundario",
    "material_sin_superficie",
    "implemento_textil_no_analizable",
    "imagen_no_util",
    "fragmento_no_legible",
    "metadata_insuficiente",
}

audit["decision_auditoria"] = audit["decision_auditoria"].fillna("").str.strip()
audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").str.strip()

errores_decision = audit[
    (audit["decision_auditoria"] != "") &
    (~audit["decision_auditoria"].isin(DECISIONES_VALIDAS))
]

errores_motivo = audit[
    (audit["motivo_auditoria"] != "") &
    (~audit["motivo_auditoria"].isin(MOTIVOS_VALIDOS))
]

pendientes = audit[
    (audit["decision_auditoria"] == "") |
    (audit["motivo_auditoria"] == "")
]

print("Total registros:", len(audit))
print("Completos:", len(audit) - len(pendientes))
print("Pendientes:", len(pendientes))
print("Errores en decisión:", len(errores_decision))
print("Errores en motivo:", len(errores_motivo))

display(errores_decision)
display(errores_motivo)
display(pendientes.head(20))

In [ ]:
# ============================================================
# Guardar auditoría validada en CSV
# ============================================================

audit_csv = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar_validada.csv"
audit.to_csv(audit_csv, index=False, encoding="utf-8-sig")

print(audit_csv)

In [ ]:
# ============================================================
# Cargar auditoría completada y validar decision_auditoria
# ============================================================

audit_path = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar.xlsx"
audit = pd.read_excel(audit_path)

audit["decision_auditoria"] = (
    audit["decision_auditoria"]
    .fillna("")
    .astype(str)
    .str.strip()
)

DECISIONES_VALIDAS = {
    "pasar_a_corpus_principal",
    "pasar_a_corpus_secundario",
    "descartar",
    "requiere_revision",
}

errores_decision = audit[
    ~audit["decision_auditoria"].isin(DECISIONES_VALIDAS)
]

pendientes_decision = audit[
    audit["decision_auditoria"] == ""
]

print("Total registros auditados:", len(audit))
print("Decisiones válidas:", len(audit) - len(errores_decision))
print("Decisiones con error:", len(errores_decision))
print("Pendientes:", len(pendientes_decision))

print("\nDistribución de decision_auditoria:")
display(
    audit["decision_auditoria"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "decision_auditoria", "decision_auditoria": "conteo"})
)

display(errores_decision)

In [ ]:
# ============================================================
# Completar motivo_auditoria sugerido según decisión y categoría
# ============================================================

def sugerir_motivo_auditoria(row):
    decision = row.get("decision_auditoria", "")
    categoria = row.get("categoria_revision_v2", "")
    titulo = str(row.get("titulo_original", "")).lower()
    nombre = str(row.get("nombre_objeto_original", "")).lower()

    if decision == "descartar":
        if categoria == "descartar_material_sin_superficie":
            return "material_sin_superficie"
        if categoria == "descartar_implemento_textil":
            return "implemento_textil_no_analizable"
        return "imagen_no_util"

    if decision == "pasar_a_corpus_secundario":
        if categoria == "secundario_featherwork":
            return "featherwork_secundario"
        if categoria == "secundario_artefacto_textil":
            return "artefacto_textil_secundario"
        if categoria in ["revision_prenda_iconografica_prioritaria", "revision_prenda_accesorio"]:
            return "prenda_tridimensional_util"
        if categoria == "revision_formato_longitudinal":
            return "faja_o_banda_util_secundario"
        return "objeto_textil_secundario"

    if decision == "pasar_a_corpus_principal":
        if "tunic" in titulo or "tunic" in nombre:
            return "superficie_textil_util"
        if "mantle" in titulo or "mantle" in nombre:
            return "superficie_textil_util"
        if "bag" in titulo or "bag" in nombre:
            return "superficie_textil_util"
        if "panel" in titulo or "panel" in nombre:
            return "superficie_textil_util"
        if "fragment" in titulo or "fragment" in nombre:
            return "fragmento_textil_util"
        if "border" in titulo or "border" in nombre:
            return "iconografia_legible"
        return "composicion_visual_util"

    if decision == "requiere_revision":
        return "metadata_insuficiente"

    return ""

# Solo completa motivo_auditoria si está vacío
if "motivo_auditoria" not in audit.columns:
    audit["motivo_auditoria"] = ""

audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").astype(str).str.strip()

audit.loc[
    audit["motivo_auditoria"] == "",
    "motivo_auditoria"
] = audit.loc[
    audit["motivo_auditoria"] == ""
].apply(sugerir_motivo_auditoria, axis=1)

display(
    audit[
        [
            "id_objeto",
            "titulo_original",
            "categoria_revision_v2",
            "decision_auditoria",
            "motivo_auditoria",
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# Validar motivos de auditoría
# ============================================================

MOTIVOS_VALIDOS = {
    "superficie_textil_util",
    "fragmento_textil_util",
    "iconografia_legible",
    "composicion_visual_util",
    "iconografia_wari_tiwanaku_relevante",
    "patron_geometrico_legible",
    "prenda_tridimensional_util",
    "prenda_tridimensional_no_prioritaria",
    "formato_longitudinal_con_motivos",
    "formato_longitudinal_sin_motivos",
    "faja_o_banda_util_secundario",
    "featherwork_secundario",
    "artefacto_textil_secundario",
    "objeto_textil_secundario",
    "material_sin_superficie",
    "implemento_textil_no_analizable",
    "imagen_no_util",
    "fragmento_no_legible",
    "metadata_insuficiente",
}

errores_motivo = audit[
    ~audit["motivo_auditoria"].isin(MOTIVOS_VALIDOS)
]

print("Total registros:", len(audit))
print("Motivos válidos:", len(audit) - len(errores_motivo))
print("Motivos con error:", len(errores_motivo))

print("\nDistribución de motivo_auditoria:")
display(
    audit["motivo_auditoria"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "motivo_auditoria", "motivo_auditoria": "conteo"})
)

display(errores_motivo)

In [ ]:
# ============================================================
# Guardar auditoría con decisiones y motivos completados
# ============================================================

audit_csv = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar_validada.csv"
audit_xlsx = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar_validada.xlsx"

audit.to_csv(audit_csv, index=False, encoding="utf-8-sig")
audit.to_excel(audit_xlsx, index=False)

print("CSV:", audit_csv)
print("Excel:", audit_xlsx)

In [ ]:
# ============================================================
# Cargar auditoría validada
# ============================================================

from pathlib import Path
import pandas as pd

audit_csv = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar_validada.csv"
audit_xlsx = ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar_validada.xlsx"

if audit_csv.exists():
    audit = pd.read_csv(audit_csv)
elif audit_xlsx.exists():
    audit = pd.read_excel(audit_xlsx)
else:
    audit = pd.read_excel(ROOT / "outputs/archive/review/auditoria_aceptados_no_preliminar.xlsx")

audit["decision_auditoria"] = audit["decision_auditoria"].fillna("").astype(str).str.strip()
audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").astype(str).str.strip()

print("Registros auditados:", len(audit))

display(
    audit["decision_auditoria"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "decision_auditoria", "decision_auditoria": "conteo"})
)

In [ ]:
# ============================================================
# Separar registros auditados según decisión final
# ============================================================

audit_principal = audit[audit["decision_auditoria"] == "pasar_a_corpus_principal"].copy()
audit_secundario = audit[audit["decision_auditoria"] == "pasar_a_corpus_secundario"].copy()
audit_descartado = audit[audit["decision_auditoria"] == "descartar"].copy()
audit_revision = audit[audit["decision_auditoria"] == "requiere_revision"].copy()

print("Pasan a corpus principal:", len(audit_principal))
print("Pasan a corpus secundario:", len(audit_secundario))
print("Descartados:", len(audit_descartado))
print("Requieren revisión:", len(audit_revision))

In [ ]:
# ============================================================
# Cargar corpus actuales
# ============================================================

corpus_clean_actual = pd.read_csv(ROOT / "data/interim/met_curacion_previa/corpus_principal_previo.csv")
corpus_secondary_actual = pd.read_csv(ROOT / "data/interim/met_curacion_previa/corpus_complementario_previo.csv")

print("Corpus limpio actual:", len(corpus_clean_actual))
print("Corpus secundario actual:", len(corpus_secondary_actual))

In [ ]:
# ============================================================
# Preparar registros auditados para integrarlos al corpus
# ============================================================

def preparar_auditados(df, destino):
    result = df.copy()

    result["origen_curacion"] = "auditoria_aceptados_no_preliminar"
    result["decision_curacion_final"] = result["decision_auditoria"]
    result["motivo_curacion_final"] = result["motivo_auditoria"]
    result["destino_curacion_final"] = destino

    if "observacion_auditoria" not in result.columns:
        result["observacion_auditoria"] = ""

    return result

audit_principal_ready = preparar_auditados(audit_principal, "corpus_principal")
audit_secundario_ready = preparar_auditados(audit_secundario, "corpus_secundario")
audit_descartado_ready = preparar_auditados(audit_descartado, "descartado")
audit_revision_ready = preparar_auditados(audit_revision, "requiere_revision")

In [ ]:
# ============================================================
# Unificar columnas entre corpus actual y registros auditados
# ============================================================

def unir_columnas(df1, df2):
    all_cols = list(dict.fromkeys(list(df1.columns) + list(df2.columns)))

    df1_aligned = df1.copy()
    df2_aligned = df2.copy()

    for col in all_cols:
        if col not in df1_aligned.columns:
            df1_aligned[col] = ""
        if col not in df2_aligned.columns:
            df2_aligned[col] = ""

    return df1_aligned[all_cols], df2_aligned[all_cols]

clean_actual_ready = corpus_clean_actual.copy()
clean_actual_ready["origen_curacion"] = "corpus_principal_previo"
clean_actual_ready["destino_curacion_final"] = "corpus_principal"

secondary_actual_ready = corpus_secondary_actual.copy()
secondary_actual_ready["origen_curacion"] = "corpus_complementario_previo"
secondary_actual_ready["destino_curacion_final"] = "corpus_secundario"

clean_actual_ready, audit_principal_ready = unir_columnas(
    clean_actual_ready,
    audit_principal_ready
)

secondary_actual_ready, audit_secundario_ready = unir_columnas(
    secondary_actual_ready,
    audit_secundario_ready
)

In [ ]:
# ============================================================
# Construir corpus corregidos v2
# ============================================================

corpus_principal_v2 = pd.concat(
    [clean_actual_ready, audit_principal_ready],
    ignore_index=True
)

corpus_secondary_v2 = pd.concat(
    [secondary_actual_ready, audit_secundario_ready],
    ignore_index=True
)

corpus_descartados_auditoria = pd.concat(
    [audit_descartado_ready, audit_revision_ready],
    ignore_index=True
)

# Evitar duplicados por id_objeto
corpus_principal_v2 = corpus_principal_v2.drop_duplicates(subset=["id_objeto"], keep="first")
corpus_secondary_v2 = corpus_secondary_v2.drop_duplicates(subset=["id_objeto"], keep="first")
corpus_descartados_auditoria = corpus_descartados_auditoria.drop_duplicates(subset=["id_objeto"], keep="first")

print("Corpus principal v2:", len(corpus_principal_v2))
print("Corpus secundario v2:", len(corpus_secondary_v2))
print("Descartados/revisión auditoría:", len(corpus_descartados_auditoria))

In [ ]:
# ============================================================
# Validar cruces entre corpus principal y secundario
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

ids_principal = set(corpus_principal_v2["id_objeto"].apply(norm_id))
ids_secundario = set(corpus_secondary_v2["id_objeto"].apply(norm_id))
ids_descartados = set(corpus_descartados_auditoria["id_objeto"].apply(norm_id))

cruce_principal_secundario = ids_principal & ids_secundario
cruce_principal_descartados = ids_principal & ids_descartados
cruce_secundario_descartados = ids_secundario & ids_descartados

print("Cruce principal/secundario:", len(cruce_principal_secundario))
print("Cruce principal/descartados:", len(cruce_principal_descartados))
print("Cruce secundario/descartados:", len(cruce_secundario_descartados))

print("IDs cruzados principal/secundario:", cruce_principal_secundario)

In [ ]:
# ============================================================
# Guardar corpus corregidos v2
# ============================================================

out_dir = ROOT / "data/metadata"

path_principal_v2 = out_dir / "corpus_principal_previo_v2.csv"
path_secondary_v2 = out_dir / "corpus_complementario_previo_v2.csv"
path_descartados = out_dir / "corpus_preliminar_descartados_auditoria.csv"

corpus_principal_v2.to_csv(path_principal_v2, index=False, encoding="utf-8-sig")
corpus_secondary_v2.to_csv(path_secondary_v2, index=False, encoding="utf-8-sig")
corpus_descartados_auditoria.to_csv(path_descartados, index=False, encoding="utf-8-sig")

print(path_principal_v2)
print(path_secondary_v2)
print(path_descartados)

In [ ]:
# ============================================================
# Resumen antes/después de auditoría
# ============================================================

resumen_auditoria = pd.DataFrame([
    {
        "etapa": "corpus_principal_actual",
        "registros": len(corpus_clean_actual),
    },
    {
        "etapa": "recuperados_para_principal",
        "registros": len(audit_principal),
    },
    {
        "etapa": "corpus_principal_v2",
        "registros": len(corpus_principal_v2),
    },
    {
        "etapa": "corpus_secundario_actual",
        "registros": len(corpus_secondary_actual),
    },
    {
        "etapa": "recuperados_para_secundario",
        "registros": len(audit_secundario),
    },
    {
        "etapa": "corpus_secundario_v2",
        "registros": len(corpus_secondary_v2),
    },
    {
        "etapa": "descartados_por_auditoria",
        "registros": len(audit_descartado),
    },
    {
        "etapa": "requieren_revision",
        "registros": len(audit_revision),
    },
])

display(resumen_auditoria)

resumen_path = ROOT / "outputs/reports/resumen_auditoria_filtros_met.csv"
resumen_path.parent.mkdir(parents=True, exist_ok=True)
resumen_auditoria.to_csv(resumen_path, index=False, encoding="utf-8-sig")

print(resumen_path)

In [ ]:
# ============================================================
# Mostrar resumen antes/después de auditoría
# ============================================================

resumen_auditoria = pd.DataFrame([
    {"etapa": "corpus_principal_actual", "registros": len(corpus_clean_actual)},
    {"etapa": "recuperados_para_principal", "registros": len(audit_principal)},
    {"etapa": "corpus_principal_v2", "registros": len(corpus_principal_v2)},
    {"etapa": "corpus_secundario_actual", "registros": len(corpus_secondary_actual)},
    {"etapa": "recuperados_para_secundario", "registros": len(audit_secundario)},
    {"etapa": "corpus_secundario_v2", "registros": len(corpus_secondary_v2)},
    {"etapa": "descartados_por_auditoria", "registros": len(audit_descartado)},
    {"etapa": "requieren_revision", "registros": len(audit_revision)},
])

display(resumen_auditoria)

resumen_path = ROOT / "outputs/reports/resumen_auditoria_filtros_met.csv"
resumen_path.parent.mkdir(parents=True, exist_ok=True)
resumen_auditoria.to_csv(resumen_path, index=False, encoding="utf-8-sig")

print(resumen_path)

In [ ]:
# ============================================================
# Validar cruces entre corpus principal, secundario y descartados
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

ids_principal = set(corpus_principal_v2["id_objeto"].apply(norm_id))
ids_secundario = set(corpus_secondary_v2["id_objeto"].apply(norm_id))
ids_descartados = set(corpus_descartados_auditoria["id_objeto"].apply(norm_id))

cruce_principal_secundario = ids_principal & ids_secundario
cruce_principal_descartados = ids_principal & ids_descartados
cruce_secundario_descartados = ids_secundario & ids_descartados

print("Cruce principal/secundario:", len(cruce_principal_secundario))
print("Cruce principal/descartados:", len(cruce_principal_descartados))
print("Cruce secundario/descartados:", len(cruce_secundario_descartados))

if cruce_principal_secundario:
    print("IDs cruzados principal/secundario:", cruce_principal_secundario)

if cruce_principal_descartados:
    print("IDs cruzados principal/descartados:", cruce_principal_descartados)

if cruce_secundario_descartados:
    print("IDs cruzados secundario/descartados:", cruce_secundario_descartados)

In [ ]:
# ============================================================
# Guardar corpus corregidos v2
# ============================================================

out_dir = ROOT / "data/metadata"

path_principal_v2 = out_dir / "corpus_principal_previo_v2.csv"
path_secondary_v2 = out_dir / "corpus_complementario_previo_v2.csv"
path_descartados = out_dir / "corpus_preliminar_descartados_auditoria.csv"

corpus_principal_v2.to_csv(path_principal_v2, index=False, encoding="utf-8-sig")
corpus_secondary_v2.to_csv(path_secondary_v2, index=False, encoding="utf-8-sig")
corpus_descartados_auditoria.to_csv(path_descartados, index=False, encoding="utf-8-sig")

print(path_principal_v2)
print(path_secondary_v2)
print(path_descartados)

In [ ]:
# ============================================================
# Generar reporte metodológico de auditoría - versión corregida
# ============================================================

report_path = ROOT / "outputs/reports/reporte_auditoria_filtros_met.md"

n_corpus_clean_actual = len(corpus_clean_actual)
n_audit_principal = len(audit_principal)
n_corpus_principal_v2 = len(corpus_principal_v2)
n_corpus_secondary_actual = len(corpus_secondary_actual)
n_audit_secundario = len(audit_secundario)
n_corpus_secondary_v2 = len(corpus_secondary_v2)
n_audit_descartado = len(audit_descartado)
n_audit_revision = len(audit_revision)

n_cruce_principal_secundario = len(cruce_principal_secundario)
n_cruce_principal_descartados = len(cruce_principal_descartados)
n_cruce_secundario_descartados = len(cruce_secundario_descartados)

report = f"""# Reporte de auditoría de filtros - The Met

## Objetivo

Revisar visualmente los registros aceptados por el filtro textil amplio que no habían pasado al corpus preliminar, con el fin de identificar falsos descartes y mejorar la trazabilidad de los motivos de curación.

## Hallazgo principal

El filtro textil amplio funcionó como primera etapa de selección, pero el filtro morfológico hacia el corpus preliminar fue demasiado restrictivo.

## Resultados antes/después

| Etapa | Registros |
|---|---:|
| Corpus principal actual | {n_corpus_clean_actual} |
| Recuperados para corpus principal | {n_audit_principal} |
| Corpus principal v2 | {n_corpus_principal_v2} |
| Corpus secundario actual | {n_corpus_secondary_actual} |
| Recuperados para corpus secundario | {n_audit_secundario} |
| Corpus secundario v2 | {n_corpus_secondary_v2} |
| Descartados por auditoría | {n_audit_descartado} |
| Requieren revisión | {n_audit_revision} |

## Validación de consistencia

| Cruce evaluado | Registros cruzados |
|---|---:|
| Principal / Secundario | {n_cruce_principal_secundario} |
| Principal / Descartados | {n_cruce_principal_descartados} |
| Secundario / Descartados | {n_cruce_secundario_descartados} |

## Interpretación metodológica

La auditoría permitió recuperar registros textiles que habían sido excluidos por criterios morfológicos demasiado restrictivos. En particular, se recuperaron fragmentos textiles, mantos, túnicas, paneles, bolsas y piezas con iconografía legible.

El corpus principal queda reservado para piezas con superficie textil visualmente analizable. El corpus secundario conserva textiles relevantes pero con morfología, función o materialidad especial, como prendas tridimensionales, formatos longitudinales, artefactos textiles y featherwork.

## Archivos generados

- `data/processed/corpus_met_textiles_andinos_v1_principal.csv`
- `data/processed/corpus_met_textiles_andinos_v1_complementario.csv`
- `data/processed/corpus_met_textiles_andinos_v1_exclusiones_curatoriales.csv`
- `outputs/archive/review/auditoria_aceptados_no_preliminar_validada.csv`
- `outputs/reports/resumen_auditoria_filtros_met.csv`

## Conclusión

La versión v2 mejora la cobertura del corpus The Met y conserva trazabilidad sobre cada decisión de curación. Esta versión debe usarse como base para la siguiente etapa de revisión visual, anotación manual y experimentos multimodales.
"""

report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report, encoding="utf-8")

print(report_path)